In [ ]:
!pip install -q transformers scipy

import cv2
import torch
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
from PIL import Image
from scipy.spatial.distance import directed_hausdorff

# ================= CONFIGURATION =================
MODEL_PATH = "/kaggle/input/datasets/gonoszgonosz/final-dice/final_dice"

INPUT_VIDEOS = [
    "/kaggle/input/rat-test-video/test.mp4",
    "/kaggle/input/rat-test-video/test2.mp4",
    "/kaggle/input/rat-test-video/test3.mp4",
]
OUTPUT_VIDEOS = [
    "/kaggle/working/b3_1024_tracking_1.mp4",
    "/kaggle/working/b3_1024_tracking_2.mp4",
    "/kaggle/working/b3_1024_tracking_3.mp4",
]

TEST_IMG_DIR  = "/kaggle/input/YOUR_TEST_DATASET/test/images"   # <--- UPDATE
TEST_MASK_DIR = "/kaggle/input/YOUR_TEST_DATASET/test/masks"    # <--- UPDATE
OUTPUT_CSV    = "/kaggle/working/b3_dice_metrics.csv"

CONFIDENCE        = 0.5
SMOOTHING_ALPHA   = 0.7
BOUNDARY_DILATION = 7
# =================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")

print("Loading model...")
model     = SegformerForSemanticSegmentation.from_pretrained(MODEL_PATH).to(device).eval()
processor = SegformerImageProcessor.from_pretrained(MODEL_PATH)
print("Model loaded.")


def infer_frame(frame, prev_prob=None):
    """Square-pad → infer → smooth → threshold → Highlander rule."""
    old_h, old_w = frame.shape[:2]
    side = max(old_h, old_w)
    canvas = np.zeros((side, side, 3), dtype=np.uint8)
    y_off = (side - old_h) // 2
    x_off = (side - old_w) // 2
    canvas[y_off:y_off + old_h, x_off:x_off + old_w] = frame

    pil = Image.fromarray(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    inputs = processor(images=pil, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = torch.nn.functional.interpolate(
            model(**inputs).logits, size=(side, side), mode="bilinear", align_corners=False
        )
        rat_prob = torch.nn.functional.softmax(logits, dim=1)[0, 1]

    smoothed = rat_prob if prev_prob is None else SMOOTHING_ALPHA * rat_prob + (1 - SMOOTHING_ALPHA) * prev_prob
    mask_square = (smoothed > CONFIDENCE).cpu().numpy().astype(np.uint8)
    mask = mask_square[y_off:y_off + old_h, x_off:x_off + old_w]

    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(cnts) > 0:
        c = max(cnts, key=cv2.contourArea)
        clean = np.zeros_like(mask)
        cv2.drawContours(clean, [c], -1, 1, thickness=cv2.FILLED)
        mask = clean
    else:
        mask = np.zeros_like(mask)

    return mask, smoothed


# ── VIDEO MASK GENERATION ──────────────────────────────────────────────────
for input_video, output_video in zip(INPUT_VIDEOS, OUTPUT_VIDEOS):
    if not os.path.exists(input_video):
        print(f"SKIPPED (not found): {input_video}")
        continue

    cap    = cv2.VideoCapture(input_video)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"\nProcessing: {input_video}  ({width}x{height}, {total} frames)")

    out      = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    prev_prob = None
    count     = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        mask, prev_prob = infer_frame(frame, prev_prob)
        green = np.zeros_like(frame); green[:, :] = [0, 255, 0]
        blended = cv2.addWeighted(frame, 0.6, green, 0.4, 0)
        frame[mask == 1] = blended[mask == 1]
        out.write(frame)
        count += 1
        if count % 100 == 0: print(f"  {count}/{total} frames...")

    cap.release(); out.release()
    print(f"Saved: {output_video}")


# ── TEST SET EVALUATION ────────────────────────────────────────────────────
def calculate_iou(pred, gt):
    inter = np.logical_and(pred, gt).sum(); union = np.logical_or(pred, gt).sum()
    if union == 0: return 1.0 if inter == 0 else 0.0
    return inter / union

def calculate_dice(pred, gt):
    inter = np.logical_and(pred, gt).sum(); total = pred.sum() + gt.sum()
    if total == 0: return 1.0
    return (2. * inter) / total

def calculate_boundary_iou(pred, gt, dilation=BOUNDARY_DILATION):
    kernel = np.ones((dilation, dilation), dtype=np.uint8)
    gt_b   = cv2.morphologyEx(gt.astype(np.uint8),   cv2.MORPH_GRADIENT, kernel) > 0
    pred_b = cv2.morphologyEx(pred.astype(np.uint8), cv2.MORPH_GRADIENT, kernel) > 0
    inter  = np.logical_and(pred_b, gt_b).sum(); union = np.logical_or(pred_b, gt_b).sum()
    if union == 0: return 1.0 if inter == 0 else 0.0
    return inter / union

def calculate_hausdorff(pred, gt):
    pred_pts = np.argwhere(cv2.Canny((pred.astype(np.uint8) * 255), 0, 1) > 0)
    gt_pts   = np.argwhere(cv2.Canny((gt.astype(np.uint8)   * 255), 0, 1) > 0)
    if len(pred_pts) == 0 or len(gt_pts) == 0: return np.nan
    return max(directed_hausdorff(pred_pts, gt_pts)[0], directed_hausdorff(gt_pts, pred_pts)[0])

img_map  = {os.path.splitext(f)[0]: f for f in os.listdir(TEST_IMG_DIR)  if f.endswith(('.jpg', '.png'))}
mask_map = {os.path.splitext(f)[0]: f for f in os.listdir(TEST_MASK_DIR) if f.endswith(('.jpg', '.png'))}
common_ids = sorted(set(img_map.keys()) & set(mask_map.keys()))
print(f"\nEvaluating on {len(common_ids)} test pairs...")

rows = []
for cid in tqdm(common_ids, desc="B3-1024 eval"):
    image   = cv2.imread(os.path.join(TEST_IMG_DIR,  img_map[cid]))
    gt_gray = cv2.imread(os.path.join(TEST_MASK_DIR, mask_map[cid]), cv2.IMREAD_GRAYSCALE)
    gt      = np.where(gt_gray > 0, 1, 0).astype(bool)
    pred, _ = infer_frame(image)
    pred    = pred.astype(bool)
    rows.append({
        "Frame_ID"          : cid,
        "mIoU"              : calculate_iou(pred, gt),
        "Dice"              : calculate_dice(pred, gt),
        "Boundary_IoU"      : calculate_boundary_iou(pred, gt),
        "Hausdorff_Distance": calculate_hausdorff(pred, gt),
    })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print("\n" + "=" * 50)
print(" B3-1024 TEST SET RESULTS")
print("=" * 50)
print(f" Frames         : {len(df)}")
print(f" Mean mIoU      : {df['mIoU'].mean():.4f}")
print(f" Mean Dice      : {df['Dice'].mean():.4f}")
print(f" Mean B-IoU     : {df['Boundary_IoU'].mean():.4f}")
print(f" Mean Hausdorff : {df['Hausdorff_Distance'].mean():.2f} px")
print(f" Saved to       : {OUTPUT_CSV}")
print("=" * 50)